# Scrape Google Trends for Your Own Topic - Lane A (the real experience: you prompt, the agent builds)

**SISMID 2026 - Day 1, 3:30 session.** You drive a coding agent (Codex, Claude Code, or
Antigravity CLI) to pull Google Trends for **your own** disease and place.

> We use the **Google Trends API** rather than scraping. Same normalized 0-100 signal as
> `pytrends`, but from a documented API: no HTTP 429s, and the same numbers every pull.
> Unlock the key first in the terminal: `source scripts/unlock-gt-api-key.sh` (Google Trends passcode),
> which exports `GT_API`. Never paste a key into a notebook you will share.


## Step 0: build a reliable fetch, and pick your topic

> *Write `gt_api_fetch(terms, geo, start, end)` that calls the Google Trends API endpoint*
> *`https://www.googleapis.com/trends/v1beta/graph` with query parameters `key`, repeated*
> *`terms`, `restrictions.geo`, `restrictions.startDate` and `restrictions.endDate`*
> *(dates are YYYY-MM). Read the key from the GT_API environment variable, never hardcode*
> *it. Parse the JSON `lines[].points[]` into a tidy DataFrame: a date column plus one*
> *column per term. Add a `gt_fetch` wrapper that falls back to the cached CSV in data/*
> *if the key is missing. Then set MY_TERMS and MY_GEO for me to edit.*


In [ ]:
from pytrends.request import TrendReq   # pip install pytrends (in requirements.txt)
import pandas as pd, os, time, random

# ===== EDIT THESE TWO LINES for your own topic =====
MY_TERMS = ['dengue', 'mosquito', 'space spraying']   # a handful of query phrases, max 5
MY_GEO   = 'US'                            # e.g. 'US', 'MX', 'US-GA', 'BR'
# ===================================================

# a cached example (flu, US) so this notebook still runs if the live pull is blocked
CACHE_PATHS = [
    '../data/google_trends_flu_us_cached.csv',
    'data/google_trends_flu_us_cached.csv',
    './google_trends_flu_us_cached.csv',
]

def _norm(c):
    return c.strip().replace(' ', '_')

def gt_fetch(kw_list, timeframe, geo, tries=4):
    """Google Trends interest-over-time for a handful of terms/topics (max 5).
    kw_list entries can be literal phrases, additive queries like 'flu + gripe',
    or topic mids like '/m/0cycc'. Returns a tidy DataFrame (date + one column
    per entry), or None if Google keeps refusing. A first 429 is normal even from
    a Codespace (Azure IP); we wait and retry. The small random pause staggers the
    room so we do not all hit Google at once."""
    time.sleep(random.uniform(0, 3))   # stagger: do NOT count '3-2-1-everyone run'
    for attempt in range(tries):
        try:
            pt = TrendReq(hl='en-US', tz=360, retries=2, backoff_factor=0.5)
            pt.build_payload(kw_list, timeframe=timeframe, geo=geo)
            df = pt.interest_over_time()
            if df.empty:
                raise RuntimeError('empty frame (rate-limited)')
            df = df.drop(columns=[c for c in ['isPartial'] if c in df.columns]).reset_index()
            return df.rename(columns={c: _norm(c) for c in df.columns})
        except Exception as e:
            rate_limited = ('429' in str(e) or 'empty frame' in str(e)
                            or 'toomany' in type(e).__name__.lower())
            if rate_limited and attempt < tries - 1:
                wait = 10 * (attempt + 1)
                print(f'Rate-limited (attempt {attempt+1}/{tries}); waiting {wait}s and retrying...')
                time.sleep(wait)
                continue
            print(f'Live Google Trends pull failed ({type(e).__name__}): {e}')
            return None
    return None

def topic_mid(phrase):
    """Resolve a phrase to a Google Trends TOPIC entity (a Knowledge Graph 'mid'
    like '/m/0cycc') via pytrends suggestions. Returns (mid, title, type); falls
    back to the literal phrase if lookup fails."""
    try:
        s = TrendReq(hl='en-US', tz=360).suggestions(phrase)
        if s:
            return s[0]['mid'], s[0]['title'], s[0]['type']
    except Exception as e:
        print('suggestions lookup failed:', e)
    return phrase, phrase, 'raw term'

def load_cache():
    for p in CACHE_PATHS:
        if os.path.exists(p):
            print(f'Using cached example (flu, US): {p}')
            return pd.read_csv(p, parse_dates=['date'])
    raise FileNotFoundError('Cached example not found; check the data/ folder.')


## Step 1: pull your topic

> *Use gt_fetch to pull MY_TERMS for MY_GEO over about the last five years. Plot the*
> *series, print the date range, and tell me whether Google returned weekly or monthly*
> *points and why.*

**Your check:** does the last point land near today, and do the peaks fall in a plausible
season for your disease and place?


In [2]:
# Agent's pull:


## Step 2: term vs topic (the language lesson)

> *The API accepts a literal term or a Knowledge Graph topic id. For each of US, FR, IT and*
> *MX, pull 'flu' and '/m/0cycc' (Influenza) in one call and tabulate the max and mean of*
> *each. Then plot both series for France.*

**What you should see:** the English term is nearly silent outside the US, while the topic
tracks the flu season everywhere, because it covers *grippe*, *influenza* and *gripe* alike.
That is why cross-country work uses topics.


In [3]:
# Agent's term-vs-topic comparison:


## Step 3: prove it is reproducible

> *Pull the same series twice and check whether the two results are identical.*

**Why it matters:** `pytrends` reads a *sampled* public endpoint, so repeat pulls
disagree and it rate-limits you. The API is deterministic, which is what makes an
analysis reproducible.


In [4]:
# Agent's reproducibility check:


## Step 4: sanity-check and save

> *Report the geo, date range, row count, resolution, missing values and percent zeros*
> *per column. Then save the tidy table to my_topic_search.csv.*


In [5]:
# Agent's checks + save:


## Reflection

- Same signal as `pytrends`, delivered reliably and reproducibly.
- **Quota and etiquette:** 1,000 requests/day on the class key, health-related topics
  only, and never commit the key.
- **Day 2:** the same loop for Wikipedia, wastewater, mobility, weather and news.
